In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# הגדרת ניהול רעש עם Estimator

<Accordion>
<AccordionItem title="גרסאות חבילות">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

יש מספר דרכים לנהל רעש, בדרך כלל על ידי שימוש בטכניקות שונות של הפחתת שגיאות ודיכוי שגיאות כדי למנוע שגיאות לפני שהן מתרחשות. טכניקות אלה גורמות בדרך כלל לעומס עיבוד מקדים. לכן, חשוב להשיג איזון בין שיפור התוצאות ובין הבטחה שהעבודה תושלם בזמן סביר.

Estimator תומך בטכניקות ניהול הרעש הבאות. ראו [טכניקות הפחתת שגיאות ודיכוי שגיאות](error-mitigation-and-suppression-techniques) להסבר על כל אחת. ראו את החלק [הגדרות שגיאה מותאמות אישית](#advanced-error) להוראות כיצד להפעיל טכניקות אלה.

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## רמת חוסן
ה-`resilience_level` מציין כמה חוסן לבנות כנגד שגיאות. רמות גבוהות יותר מייצרות תוצאות מדויקות יותר, במחיר של זמני עיבוד ארוכים יותר. אפשר להשתמש ברמות חוסן כדי להגדיר את הפשרה בין עלות לדיוק בעת יישום ניהול רעש לשאילתת ה-primitive שלך. ניהול רעש מפחית שגיאות (הטיה) בתוצאות על ידי עיבוד הפלטים מאוסף, או ensemble, של מעגלים קשורים. מידת הפחתת השגיאה תלויה בשיטה המיושמת. רמת החוסן מפשטת את הבחירה המפורטת של שיטת ניהול הרעש כדי לאפשר למשתמשים לחשוב על הפשרה בין עלות לדיוק שמתאימה לאפליקציה שלהם.

בהתאם לכך, כל רמה מתאימה לשיטה או שיטות עם רמת עומס דגימה קוונטית הולכת וגדלה, כדי לאפשר לך להתנסות עם פשרות שונות בין זמן לדיוק. הטבלה הבאה מציגה אילו רמות ושיטות מתאימות זמינות עבור כל אחד מה-primitives.

<span id="resilience-table"></span>

| רמת חוסן | תיאור | טכניקה |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | ללא הפחתה | ללא |
| 1 [ברירת מחדל] | עלויות הפחתה מינימליות: הפחתת שגיאות הקשורות לשגיאות קריאה | מדידת [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) twirling |
| 2 | עלויות הפחתה בינוניות. בדרך כלל מפחית הטיה ב-estimators, אבל לא מובטח שתהיה אפס הטיה. | רמה 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) ו-gate twirling

> **Info:** רמות חוסן נמצאות כעת בבטא, כך שעומס הדגימה ואיכות הפתרון ישתנו ממעגל למעגל. תכונות חדשות, אפשרויות מתקדמות וכלי ניהול ישוחררו באופן שוטף. שיטות ניהול רעש ספציפיות אינן מובטחות ליישום בכל רמת חוסן.

### הגדרת Estimator עם רמות חוסן
אפשר להשתמש ברמות חוסן כדי לציין טכניקות ניהול רעש, או להגדיר טכניקות מותאמות אישית בנפרד כמתואר ב[הגדרות שגיאה מותאמות אישית](#advanced-error).

> **Note:** כל אפשרות שמגדירים ידנית בנוסף לרמת החוסן מיושמת בנוסף לסט הבסיסי של אפשרויות שהוגדרו על ידי רמת החוסן. לכן, עקרונית, אפשר להגדיר את רמת החוסן ל-1, אך לאחר מכן לכבות את הפחתת המדידה, אם כי זה לא מומלץ.
> 
> לדוגמה, הגדרת רמת החוסן ל-0 מכבה את `zne_mitigation`, אך `estimator.options.resilience.zne_mitigation = True` מבטל ערך זה.

### דוגמה
הקוד הבא מאפשר ZNE, TREX ו-gate twirling על ידי הגדרת `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## הגדרות ניהול רעש מותאמות אישית
אפשר להפעיל ולכבות שיטות ניהול רעש בודדות על ידי שימוש ב[אפשרויות Estimator](/guides/estimator-options).

> **Note:** לא כל האפשרויות עובדות יחד על כל סוגי המעגלים. ראו את [טבלת תאימות התכונות](estimator-options#options-compatibility-table) לפרטים.

### דוגמה

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>